In [ ]:
# [Scenario 3]: Automated Farm Monitoring System
# [Tech Stack]: Edge ML, Cloud IoT, Predictive Analytics

import time
import numpy as np
import json
from datetime import datetime
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score
import joblib
import requests

class SoilHydrationReader:
    def __init__(self):
        self.calibration_factor = 0.97
    
    def measure_hydration(self):
        return np.random.uniform(25, 75)
    
    def measure_temperature(self):
        return np.random.uniform(5, 40)

class IrrigationPredictor:
    def __init__(self):
        self.model = None
        self.initialize_predictor()
    
    def initialize_predictor(self):
        try:
            self.model = joblib.load('farm_model.pkl')
            print("Loaded existing prediction model")
        except FileNotFoundError:
            self.train_new_model()
    
    def train_new_model(self):
        synthetic_data = {
            'hydration': [np.random.uniform(20, 80) for _ in range(1500)],
            'temperature': [np.random.uniform(10, 40) for _ in range(1500)],
            'time_segment': [t//6 for t in range(1500)],
            'water_needed': np.random.randint(0, 2, 1500)
        }
        
        df = pd.DataFrame(synthetic_data)
        X = df[['hydration', 'temperature', 'time_segment']]
        y = df['water_needed']
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)
        self.model = DecisionTreeClassifier(max_depth=4)
        self.model.fit(X_train, y_train)
        
        predictions = self.model.predict(X_test)
        print(f"Model trained | Precision: {precision_score(y_test, predictions):.2f}")
        joblib.dump(self.model, 'farm_model.pkl')

    def predict_irrigation(self, hydration, temp):
        time_segment = datetime.now().hour // 4
        return self.model.predict([[hydration, temp, time_segment]])[0]

class CloudManager:
    def __init__(self):
        self.api_endpoint = "https://agri-cloud.example.com/alerts"
        self.auth_token = "farm_system_token"
    
    def send_alert(self, message):
        headers = {
            "Authorization": f"Bearer {self.auth_token}",
            "Content-Type": "application/json"
        }
        payload = {
            "alert": message,
            "timestamp": datetime.utcnow().isoformat(),
            "device": "field_sensor_01"
        }
        try:
            response = requests.post(self.api_endpoint, json=payload, headers=headers)
            return response.status_code == 202
        except Exception as e:
            print(f"Cloud error: {str(e)}")
            return False

class FarmMonitor:
    def __init__(self):
        self.sensor = SoilHydrationReader()
        self.predictor = IrrigationPredictor()
        self.cloud = CloudManager()
        self.interval = 300  # 5 minutes
    
    def start(self):
        print("Farm Monitoring System Activated\n")
        while True:
            hydration = self.sensor.measure_hydration()
            temp = self.sensor.measure_temperature()
            
            print(f"Current Readings - Hydration: {hydration:.1f}%, Temp: {temp:.1f}°C")
            
            if self.predictor.predict_irrigation(hydration, temp):
                alert_msg = f"Irrigation Required: {hydration:.1f}% hydration at {temp:.1f}°C"
                self.cloud.send_alert(alert_msg)
                print("Alert sent to cloud\n")
            
            time.sleep(self.interval)

if __name__ == "__main__":
    monitor = FarmMonitor()
    monitor.start()

In [ ]:
# [Scenario 1]: Smart Home Controller
# [Tech Stack]: IoT Sensors, Secure Messaging, Local Storage

import sqlite3
import paho.mqtt.client as mqtt
import json
from time import sleep
from datetime import datetime

MAX_TEMP = 28.0
MAX_HUMIDITY = 75.0
DB_NAME = 'environment.db'
MQTT_SERVER = 'mqtt.residential.example'
DATA_TOPIC = 'home/sensors/environment'
ALERT_TOPIC = 'home/alerts/notifications'
POLL_RATE = 45

class EnvironmentSensor:
    @staticmethod
    def read_values():
        t = 22.0 + (8 * (time.time() % 15) / 15)
        h = 50.0 + (30 * (time.time() % 15) / 15)
        return {
            'temperature': round(t, 1),
            'humidity': round(h, 1),
            'timestamp': datetime.now().isoformat()
        }

def init_database():
    with sqlite3.connect(DB_NAME) as conn:
        conn.execute('''CREATE TABLE IF NOT EXISTS sensor_data
                     (id INTEGER PRIMARY KEY,
                      temperature REAL,
                      humidity REAL,
                      timestamp TEXT,
                      alerted BOOLEAN DEFAULT 0)''')

def store_data(temp, humid, ts):
    with sqlite3.connect(DB_NAME) as conn:
        conn.execute('''INSERT INTO sensor_data (temperature, humidity, timestamp)
                     VALUES (?, ?, ?)''', (temp, humid, ts))

mqtt_client = mqtt.Client()
mqtt_client.connect(MQTT_SERVER)

def on_connect(client, userdata, flags, rc):
    print("Connected to MQTT broker" if rc == 0 else "Connection failed")
    client.subscribe(ALERT_TOPIC)

def handle_message(client, userdata, msg):
    print(f"Received MQTT message: {msg.payload.decode()}")

mqtt_client.on_connect = on_connect
mqtt_client.on_message = handle_message
mqtt_client.loop_start()

def trigger_alert(temp, humid):
    alert = {
        "condition": "threshold_exceeded",
        "temperature": temp,
        "humidity": humid,
        "timestamp": datetime.utcnow().isoformat()
    }
    mqtt_client.publish(DATA_TOPIC, json.dumps(alert))
    print(f"Alert published: {temp}°C, {humid}%")

def monitor_environment():
    init_database()
    
    while True:
        try:
            data = EnvironmentSensor.read_values()
            store_data(data['temperature'], data['humidity'], data['timestamp'])
            mqtt_client.publish(DATA_TOPIC, json.dumps(data))
            
            if data['temperature'] > MAX_TEMP or data['humidity'] > MAX_HUMIDITY:
                trigger_alert(data['temperature'], data['humidity'])
            
            sleep(POLL_RATE)
            
        except Exception as e:
            print(f"Monitoring error: {str(e)}")
            sleep(10)

if __name__ == "__main__":
    monitor_environment()

## Data Processing Module

In [ ]:
// Sensor Data Processor with Secure Transmission
#include <vector>
#include <numeric>
#include <cpprest/http_client.h>

const int SAMPLE_WINDOW = 15;

float read_sensor_value() {
    return 22.5f + (std::rand() % 1000 - 500)/200.0f;
}

float calculate_avg(const std::vector<float>& values) {
    return std::accumulate(values.begin(), values.end(), 0.0f) / values.size();
}

bool transmit_data(float value) {
    using namespace web::http;
    
    client::http_client client(U("https://iot-gateway.example/data"));
    json::value payload = json::value::parse(
        "{\"sensor\":\"environment\", \"value\":" + std::to_string(value) + "}"
    );

    try {
        client.request(methods::POST, U(""), payload).wait();
        return true;
    } catch (...) {
        return false;
    }
}

int main() {
    std::vector<float> data_buffer;
    while(true) {
        data_buffer.push_back(read_sensor_value());
        
        if(data_buffer.size() >= SAMPLE_WINDOW) {
            float avg = calculate_avg(data_buffer);
            std::cout << "Transmitting average: " << avg << "°C\n";
            
            if(!transmit_data(avg)) {
                std::cerr << "Data transmission failed!\n";
            }
            
            data_buffer.clear();
        }
        
        std::this_thread::sleep_for(std::chrono::seconds(2));
    }
    return 0;
}